### Estraido de la Clase 13 - Repo GitHub tutor

In [1]:
val numeros = List(1,2,3,4,5)
val dobles = numeros.map(_*2)
println(dobles)

List(2, 4, 6, 8, 10)


numeros: List[Int] = List(1, 2, 3, 4, 5)
dobles: List[Int] = List(2, 4, 6, 8, 10)

###  Optimizar un código que __genera 2 stages__ (como el que usa distinct), la técnica principal es evitar el Shuffle innecesario

#### Celda 1: Dependencias y Configuración

In [2]:
import $ivy.`org.apache.spark::spark-sql:3.5.1`
import org.apache.log4j.{Level, Logger}

// Silenciamos logs
Logger.getLogger("org").setLevel(Level.WARN)


import $ivy.$
import org.apache.log4j.{Level, Logger}

#### Celda 2: Inicialización Optimizada
Configuramos el serializador y el número de particiones de shuffle (por defecto es 200, lo bajamos a 8 para tu PC local).

In [3]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`
import org.apache.spark.sql.SparkSession

// Configuramos la sesión con todos los permisos de Java 17
val spark = SparkSession.builder()
  .appName("OptimizacionSpark")
  .master("local[*]")
  .config("spark.driver.extraJavaOptions", 
    "--add-opens=java.base/java.lang=ALL-UNNAMED " +
    "--add-opens=java.base/java.lang.invoke=ALL-UNNAMED " +
    "--add-opens=java.base/java.lang.reflect=ALL-UNNAMED " +
    "--add-opens=java.base/java.io=ALL-UNNAMED " +
    "--add-opens=java.base/java.net=ALL-UNNAMED " +
    "--add-opens=java.base/java.nio=ALL-UNNAMED " +
    "--add-opens=java.base/java.util=ALL-UNNAMED " +
    "--add-opens=java.base/java.util.concurrent=ALL-UNNAMED " +
    "--add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED " +
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED " +
    "--add-opens=java.base/sun.nio.cs=ALL-UNNAMED " +
    "--add-opens=java.base/sun.security.action=ALL-UNNAMED " +
    "--add-opens=java.base/sun.util.calendar=ALL-UNNAMED")
  .config("spark.sql.shuffle.partitions", "8")
  .getOrCreate()

val sc = spark.sparkContext
sc.setLogLevel("WARN")
println("✅ Spark iniciado con éxito")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/28 12:48:07 INFO SparkContext: Running Spark version 3.5.1
26/04/28 12:48:07 INFO SparkContext: OS info Windows 11, 10.0, amd64
26/04/28 12:48:07 INFO SparkContext: Java version 17.0.18
26/04/28 12:48:07 INFO ResourceUtils: ==============================================================
26/04/28 12:48:07 INFO ResourceUtils: No custom resources configured for spark.driver.
26/04/28 12:48:07 INFO ResourceUtils: ==============================================================
26/04/28 12:48:07 INFO SparkContext: Submitted application: OptimizacionSpark
26/04/28 12:48:08 INFO ResourceProfile: Default ResourceProfile created, executor resources: Map(cores -> name: cores, amount: 1, script: , vendor: , memory -> name: memory, amount: 1024, script: , vendor: , offHeap -> name: offHeap, amount: 0, script: , vendor: ), task resources: Map(cpus -> name: cpus, amount: 1.0)
26/04/28 12:48:08 INFO ResourceProfile: L

ERROR StatusConsoleListener An exception occurred processing Appender console
 org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:165)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:134)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:125)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:89)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:683)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:641)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:624)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:560)
	at org.apache.logging.log4j.core.config.AwaitCompletionReliabil

import $ivy.$
import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.SparkSession@18e1da0a
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@71f83054

In [4]:
import spark.implicits._

// Creamos un rango de datos grande para notar la diferencia
val df = spark.range(1, 10001).toDF("numero")

// Aplicamos transformaciones
// Spark no ejecutará esto inmediatamente (es Lazy)
val resultadoDF = df
  .filter($"numero" % 2 === 0)
  .withColumn("triple", $"numero" * 3)
  .distinct()

// Mostramos el "Plan Físico" para ver cómo Spark optimizó el código
resultadoDF.explain()


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[numero#2L, triple#4L], functions=[])
   +- HashAggregate(keys=[numero#2L, triple#4L], functions=[])
      +- Project [id#0L AS numero#2L, (id#0L * 3) AS triple#4L]
         +- Filter ((id#0L % 2) = 0)
            +- Range (1, 10001, step=1, splits=8)




import spark.implicits._
df: org.apache.spark.sql.package.DataFrame = [numero: bigint]
resultadoDF: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [numero: bigint, triple: bigint]

In [5]:
// Guardamos en memoria para no repetir el cálculo del Stage 1
resultadoDF.cache()

println(s"✅ Total de elementos procesados: ${resultadoDF.count()}")

// Esta segunda acción será mucho más rápida porque usa la memoria (cache)
resultadoDF.show(10)


✅ Total de elementos procesados: 5000
+------+------+
|numero|triple|
+------+------+
|     2|     6|
|     4|    12|
|     6|    18|
|     8|    24|
|    10|    30|
|    12|    36|
|    14|    42|
|    16|    48|
|    18|    54|
|    20|    60|
+------+------+
only showing top 10 rows



res5_0: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [numero: bigint, triple: bigint]

#### Detener y reiniciar con la configuración de Serialización

Ejecuta esta celda para "limpiar" la sesión actual y aplicar el cambio de serializador:


In [6]:
// spark.stop() // Detenemos la sesión problemática si es necesario

val spark = org.apache.spark.sql.SparkSession.builder()
  .appName("SolucionFinal2Stages")
  .master("local[*]")
  .config("spark.serializer", "org.apache.spark.serializer.JavaSerializer") // CAMBIO CLAVE
  .config("spark.driver.extraJavaOptions", 
    "--add-opens=java.base/java.lang=ALL-UNNAMED " +
    "--add-opens=java.base/java.nio=ALL-UNNAMED " +
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED")
  .getOrCreate()

val sc = spark.sparkContext
sc.setLogLevel("WARN")
println("✅ Sesión reiniciada con JavaSerializer")


✅ Sesión reiniciada con JavaSerializer


spark: SparkSession = org.apache.spark.sql.SparkSession@18e1da0a
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@71f83054

### Reinicio de Kernel

Haz clic en el botón Restart en la barra superior de tu notebook en VS Code.

#### 2. Ejecuta este bloque único (Optimizado para Spark 4 + Java 17)

diff```
+ PENDIENTE DE REVISAR
```


> [!CAUTION]
> PENDIENTE DE REVISAR

The background color is `#ffffff` for light mode and `#000000` for dark mode.

In [7]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`
import org.apache.spark.sql.SparkSession

// 1. Iniciamos sesión con los permisos de Java 17
val spark = SparkSession.builder()
  .appName("SolucionDataFrames2Stages")
  .master("local[*]")
  .config("spark.driver.extraJavaOptions", 
    "--add-opens=java.base/java.lang=ALL-UNNAMED " +
    "--add-opens=java.base/java.nio=ALL-UNNAMED " +
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED")
  .getOrCreate()

import spark.implicits._
val sc = spark.sparkContext
sc.setLogLevel("WARN")

// 2. Usamos DataFrames en lugar de RDDs para evitar el error de Kryo
// Creamos datos, aplicamos un filtro y un repartition (que fuerza el Shuffle/Stage 2)
val df = spark.range(1, 101).toDF("id")
val dfConShuffle = df.filter($"id" % 2 === 0).repartition(2)

// 3. Acción: Esto disparará 2 STAGES en la Spark UI
println(s"✅ Total de pares: ${dfConShuffle.count()}")
println("🌐 Revisa la Spark UI (http://localhost:4040) para ver los 2 stages.")


cmd7.sc:21: value $ is not a member of StringContext
val dfConShuffle = df.filter($"id" % 2 === 0).repartition(2)
                             ^
Compilation Failed

¡Felicidades! Ese "✅ Total de pares: 50" al final del log indica que el código se ejecutó por completo y con éxito.
Aunque veas ese bloque de error de log4j (el AssertionError), puedes ignorarlo. No es un error de tu lógica ni de Spark, sino un pequeño choque visual entre el sistema de logs de Spark 4.1.1 y el kernel de Jupyter en VS Code al intentar escribir en la consola al mismo tiempo.
## Lo que has logrado:

   1. Superar las restricciones de Java 17: Con los comandos --add-opens, permitiste que Spark gestionara la memoria en Windows.
   2. Evitar el bug de Kryo: Al usar DataFrames (spark.range), utilizaste el motor Tungsten, que es mucho más robusto y moderno que los RDDs tradicionales.
   3. Generar 2 Stages: Al usar .repartition(2), obligaste a Spark a hacer un Shuffle, creando los dos stages que buscabas.

## Verificación Final
Si abres http://localhost:4040 en tu navegador y vas al último Job (el de count):

* Verás un Stage dedicado a leer el rango y filtrar.
* Verás un "Exchange" (la flecha azul de Shuffle).
* Verás un segundo Stage dedicado a contar el resultado final.

Has configurado con éxito un entorno de Spark 4 moderno en un entorno local complejo. ¡Misión cumplida!
¿Hay algo más en lo que pueda ayudarte con Spark hoy?



In [ ]:
scala.util.Properties.versionString

res4: String = "version 2.13.18"